# CSC271H1 Week 10 Tutorial: Patterns, Parameters, and Joins

## Goals of this tutorial:
- Learn about the concatenation operator, `||`, and the absolute value, `ABS`, function.
- Practice pattern matching in queries.
- Practice writing parameterized queries.
- Practice writing queries involving joins.

## Introduction

In this tutorial, you will work with a database containing weather statistics for Canadian cities. The database has three tables, `cities`, `precipitation`, and `temperature` with the following schemas:

<pre>cities(<u>city</u>: TEXT, province: TEXT)</pre>

<pre>precipitation(<u>city</u>: TEXT, snow: FLOAT, total_precip: FLOAT, days_with_precip: INT)</pre>

<pre>temperature(
  <u>city</u>: TEXT, average_high: FLOAT, average_low: FLOAT,
  coldest_month: TEXT, avg_high_coldest_month: FLOAT, avg_low_coldest_month: FLOAT,
  warmest_month: TEXT, avg_high_warmest_month: FLOAT, avg_low_warmest_month: FLOAT)</pre>
</div>

Additionally, the database has these foreign keys:

<pre>
FK: precipitation.city -> cities.city
FK: temperature.city -> cities.city
</pre>

## Task 0: Connect to the Database

We already created the database that we'll use in this tutorial. You can either copy your local copy of the database from last week's tutorial into the same directory as this notebook or download the provided database file. Once you've done that, run the code below to connect to the database.

In [1]:
import w10_helpers
import sqlite3
import pandas as pd

conn = w10_helpers.connect('weather.sqlite')

## Task 1: Query the Database

#### A) Complete the function below according to its docstring description. 

Hint: this is similar to the first query you wrote last week, but the threshold varies.

In [2]:
def get_precip_below_threshold(conn: sqlite3.Connection, 
                                     threshold: int) -> pd.DataFrame:
    """Return the names of the cities and the number of days of precipitation for
    cities whose total annual precipitation is below threshold."""

    ### STARTER CODE ###
    query = '''
    SELECT city, days_with_precip
    FROM precipitation
    WHERE total_precip < ?
    ''' 
    
    return w10_helpers.run_sql(conn, query, (threshold,))

threshold = 300
below_threshold = get_precip_below_threshold(conn, threshold)
below_threshold

,city,days_with_precip
0,Whitehorse,122
1,Yellowknife,118


#### B) Complete the function below according to its docstring description. 

Hint: use the concatenation operator `||` and pattern matching.

In [3]:
def province_startswith(conn: sqlite3.Connection, 
                      prefix: str) -> pd.DataFrame:
    """Return the cities and provinces in the database connected to
    by conn that have province names starting with prefix."""

    ### STARTER CODE ###
    query = '''
    SELECT city, province
    FROM cities
    WHERE province LIKE ? || '%'
    '''  
    
    return w10_helpers.run_sql(conn, query, (prefix,))

prefix = 'N'
cities_provinces = province_startswith(conn, prefix)
cities_provinces

,city,province
0,St.John's,Newfoundland
1,Halifax,Nova Scotia
2,Fredericton,New Brunswick
3,Yellowknife,NWT


#### C) Complete the function below according to its docstring description.

In [ ]:
def get_same_months(conn: sqlite3.Connection, 
                    cold_month: str, warm_month: str) -> pd.DataFrame:
    """Return the cities in the database connected to by conn whose
    coldest month is cold_month and warmest month is warm_month.
    """

    ### STARTER CODE ###
    query = '''
    SELECT city
    FROM temperature
    WHERE coldest_month = ? AND warmest_month = ?
    '''  
    
    return w10_helpers.run_sql(conn, query, (cold_month, warm_month))

cold = 'January'
warm = 'July'
cities_same = get_same_months(conn, cold, warm)
cities_same

#### D) Complete the function below according to its docstring description.

In [ ]:
def get_snow_within_range(conn: sqlite3.Connection, min_snow: int, max_snow) -> pd.DataFrame:
    """Return the names of provinces in the database connected to by conn with
    cities that have snow totals between min_snow and max_snow, inclusive.
    """

    ### STARTER CODE ###
    query = '''
    SELECT DISTINCT province
    FROM cities
    JOIN precipitation ON cities.city = precipitation.city
    WHERE snow BETWEEN ? AND ?
    ''' 
    
    return w10_helpers.run_sql(conn, query, (min_snow, max_snow))

min_snow = 200
max_snow = 300
provinces = get_snow_within_range(conn, min_snow, max_snow)
provinces

#### Complete the function below according to its docstring description. 

Hint: use the `ABS` function.

Do not include self-pairs or duplicates: 
- E.g., do not include `('Charlottetown', 338.7, 'Charlottetown', 338.7)`.
- E.g., if the results include `('Charlottetown', 338.7, 'Quebec', 337.0)`, then 
don't also include `('Quebec', 337.0, 'Charlottetown', 338.7)`.

In [ ]:
def get_cities_within_threshold(conn: sqlite3.Connection, max_diff: int) -> pd.DataFrame:
    """Return the pairs of cities that have snow amounts that are less than or equal to
    max_diff of each other in the database connected to by conn.

    The result should include four columns: 1st city, 1st snow, 2nd city, 2nd snow
    """

    ### STARTER CODE ###
    query = '''
    SELECT p1.city, p1.snow, p2.city, p2.snow
    FROM precipitation p1
    JOIN precipitation p2 ON p1.city < p2.city
    WHERE ABS(p1.snow - p2.snow) <= ?
    ''' 

    return w10_helpers.run_sql(conn, query, (max_diff,))

diff = 5
city_pairs = get_cities_within_threshold(conn, diff)
city_pairs

### Final Task: Show your TA and Submit to MarkUs

If you have not already done so, make sure your TA has recorded your attendance.

Submit your completed `w010_tutorial.ipynb` file to [MarkUs](https://markus.teach.cs.toronto.edu/markus/courses/128) and run the tests.